<a href="https://colab.research.google.com/github/doomguy0991/dls_c/blob/main/C2%20-%20Improving%20Deep%20Neural%20Networks%20Hyperparameter%20tuning%2C%20Regularization%20and%20Optimization/Notes/Readme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Setting up Colab environment...")

    # 1. Clone the repository
    repo_url = "https://github.com/doomguy0991/dls_c.git"
    # Only clone if the directory doesn't exist yet
    if not os.path.exists("/content/dls_c"):
        !git clone $repo_url /content/dls_c

    # 2. Change working directory to the notebook's location so relative paths for libraries work imports
    %cd "/content/dls_c/C2 - Improving Deep Neural Networks Hyperparameter tuning, Regularization and Optimization/Notes"

    print("Setup complete. You can now run the rest of the notebook.")
else:
    print("Running locally or out of Colab. No setup needed.")


### Improving Deep Neural Networks: Hyperparameter tuning, Regularization and Optimization

This is the second course of the deep learning specialization at [Coursera](https://www.coursera.org/specializations/deep-learning) which is moderated by [DeepLearning.ai](http://deeplearning.ai/). The course is taught by Andrew Ng.

#### Table of contents

* [Improving Deep Neural Networks: Hyperparameter tuning, Regularization and Optimization](#improving-deep-neural-networks-hyperparameter-tuning-regularization-and-optimization)
   * [Table of contents](#table-of-contents)
   * [Course summary](#course-summary)
   * [Practical aspects of Deep Learning](#practical-aspects-of-deep-learning)
      * [Train / Dev / Test sets](#train--dev--test-sets)
      * [Bias / Variance](#bias--variance)
      * [Basic Recipe for Machine Learning](#basic-recipe-for-machine-learning)
      * [Regularization](#regularization)
      * [Why regularization reduces overfitting?](#why-regularization-reduces-overfitting)
      * [Dropout Regularization](#dropout-regularization)
      * [Understanding Dropout](#understanding-dropout)
      * [Other regularization methods](#other-regularization-methods)
      * [Normalizing inputs](#normalizing-inputs)
      * [Vanishing / Exploding gradients](#vanishing--exploding-gradients)
      * [Weight Initialization for Deep Networks](#weight-initialization-for-deep-networks)
      * [Numerical approximation of gradients](#numerical-approximation-of-gradients)
      * [Gradient checking implementation notes](#gradient-checking-implementation-notes)
      * [Initialization summary](#initialization-summary)
      * [Regularization summary](#regularization-summary)
   * [Optimization algorithms](#optimization-algorithms)
      * [Mini-batch gradient descent](#mini-batch-gradient-descent)
      * [Understanding mini-batch gradient descent](#understanding-mini-batch-gradient-descent)
      * [Exponentially weighted averages](#exponentially-weighted-averages)
      * [Understanding exponentially weighted averages](#understanding-exponentially-weighted-averages)
      * [Bias correction in exponentially weighted averages](#bias-correction-in-exponentially-weighted-averages)
      * [Gradient descent with momentum](#gradient-descent-with-momentum)
      * [RMSprop](#rmsprop)
      * [Adam optimization algorithm](#adam-optimization-algorithm)
      * [Learning rate decay](#learning-rate-decay)
      * [The problem of local optima](#the-problem-of-local-optima)
   * [Hyperparameter tuning, Batch Normalization and Programming Frameworks](#hyperparameter-tuning-batch-normalization-and-programming-frameworks)
      * [Tuning process](#tuning-process)
      * [Using an appropriate scale to pick hyperparameters](#using-an-appropriate-scale-to-pick-hyperparameters)
      * [Hyperparameters tuning in practice: Pandas vs. Caviar](#hyperparameters-tuning-in-practice-pandas-vs-caviar)
      * [Normalizing activations in a network](#normalizing-activations-in-a-network)
      * [Fitting Batch Normalization into a neural network](#fitting-batch-normalization-into-a-neural-network)
      * [Why does Batch normalization work?](#why-does-batch-normalization-work)
      * [Batch normalization at test time](#batch-normalization-at-test-time)
      * [Softmax Regression](#softmax-regression)
      * [Training a Softmax classifier](#training-a-softmax-classifier)
      * [Deep learning frameworks](#deep-learning-frameworks)
      * [TensorFlow](#tensorflow)
   * [Extra Notes](#extra-notes)

## Course summary

Here are the course summary as its given on the course [link](https://www.coursera.org/learn/deep-neural-network):

> This course will teach you the "magic" of getting deep learning to work well. Rather than the deep learning process being a black box, you will understand what drives performance, and be able to more systematically get good results. You will also learn TensorFlow.
>
> After 3 weeks, you will:
> - Understand industry best-practices for building deep learning applications.
> - Be able to effectively use the common neural network "tricks", including initialization, L2 and dropout regularization, Batch normalization, gradient checking,
> - Be able to implement and apply a variety of optimization algorithms, such as mini-batch gradient descent, Momentum, RMSprop and Adam, and check for their convergence.
> - Understand new best-practices for the deep learning era of how to set up train/dev/test sets and analyze bias/variance
> - Be able to implement a neural network in TensorFlow.
>
> This is the second course of the Deep Learning Specialization.

## Practical aspects of Deep Learning

### Train / Dev / Test sets

- Its impossible to get all your hyperparameters right on a new application from the first time.
- So the idea is you go through the loop: `Idea ==> Code ==> Experiment`.
- You have to go through the loop many times to figure out your hyperparameters.
- Your data will be split into **three parts**:
  - **Training set**.       (Has to be the largest set)
  - Hold-out cross validation set / Development or **"dev" set**.
  - **Testing set**.
- You will try to **build a model upon training set** then **try to optimize hyperparameters on dev set** as much as possible. Then after your model is ready you try and **evaluate using the testing set**.
- so the trend on the ratio of splitting the models:
  - If size of the  dataset is 100 to 1000000  ==> 60/20/20
  - If size of the  dataset is 1000000  to INF  ==> 98/1/1 or  99.5/0.25/0.25
- The trend now gives the training data the biggest sets.
- Make sure the **dev** and **test set** are **coming from the same distribution**.
  - For example if cat training/dev pictures are from the web but the test pictures are from users cell phone they will mismatch. It is better to make sure that dev and test set are from the same distribution.
- The dev set rule is to try them on some of the good models you've created.
- Its OK to only have a dev set without a testing set. But a lot of people in this case call the dev set as the test set. A better terminology is to call it a dev set as its used in the development.


### Bias / Variance

- Bias / Variance techniques are Easy to learn, but difficult to master.
- So here the explanation of Bias / Variance:
  - If your model is underfitting (logistic regression of non linear data) it has a "high bias"
  - If your model is overfitting then it has a "high variance"
  - Your model will be alright if you balance the Bias / Variance
  - For more:
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C2%20-%20Improving%20Deep%20Neural%20Networks%20Hyperparameter%20tuning%2C%20Regularization%20and%20Optimization/Notes/Images/01-_Bias_-_Variance.png?raw=1)
- Another idea to get the bias /  variance if you don't have a 2D plotting mechanism:
  - High variance (overfitting) for example:
    - Training error: 1%
    - Dev error: 11%
  - high Bias (underfitting) for example:
    - Training error: 15%
    - Dev error: 14%
  - high Bias (underfitting) && High variance (overfitting) for example:
    - Training error: 15%
    - Test error: 30%
  - Best:
    - Training error: 0.5%
    - Test error: 1%
  - These Assumptions came from that **human has 0% error** ->**(Optimal/Bayes error = 0%)**. If the problem isn't like that you'll need to use human error as baseline.

### Basic Recipe for Machine Learning

- If your algorithm has a **high bias**:
  - Try to make your NN bigger (size of hidden units, number of layers)
  - Try a different model that is suitable for your data.
  - Try to run it longer.
  - Different (advanced) optimization algorithms.
- If your algorithm has a **high variance**:
  - More data.
  - Try regularization.
  - Try a different model that is suitable for your data.
- You should **try the previous two points until** you have a **low bias and low variance**.
- In the older days before deep learning, there was a "Bias/variance tradeoff". But because now you have more options/tools for solving the bias and variance problem its really helpful to use deep learning.
- Training a bigger neural network never hurts.

### Regularization

- Adding regularization to a Neural Network (NN) helps reduce variance (overfitting).

**Matrix Norms**
- **L1 Matrix Norm:**
  $$||W||_1 = \sum_{i,j} |w_{i,j}|$$
  *(The sum of the absolute values of all weights)*
- **L2 Matrix Norm** (Known as the **Frobenius norm** due to technical math conventions):
  $$||W||_F^2 = \sum_{i,j} |w_{i,j}|^2$$
  *(The sum of all weights squared)*
  - If $W$ is a vector, this can also be calculated as: $$||W||_2^2 = W^T W$$

**Regularization for Logistic Regression**
- The standard cost function we want to minimize is:
  $$J(w,b) = \frac{1}{m} \sum_{i=1}^{m} \mathcal{L}(\hat{y}^{(i)}, y^{(i)})$$
- **L2 Regularization version:**
  $$J(w,b) = \frac{1}{m} \sum_{i=1}^{m} \mathcal{L}(\hat{y}^{(i)}, y^{(i)}) + \frac{\lambda}{2m} \sum_{j=1}^{n_x} w_j^2$$
- **L1 Regularization version:**
  $$J(w,b) = \frac{1}{m} \sum_{i=1}^{m} \mathcal{L}(\hat{y}^{(i)}, y^{(i)}) + \frac{\lambda}{2m} \sum_{j=1}^{n_x} |w_j|$$
  - *Note:* L1 regularization drives many $w$ values to exactly zero, which leads to a sparser and smaller model. However, **L2 regularization** is used much more frequently in practice.
  - $\lambda$ (lambda) is the regularization hyperparameter.

**Regularization for Neural Networks**
- The standard cost function is:
  $$J(W^{[1]}, b^{[1]}, \dots, W^{[L]}, b^{[L]}) = \frac{1}{m} \sum_{i=1}^{m} \mathcal{L}(\hat{y}^{(i)}, y^{(i)})$$
- **L2 Regularization version:**
  $$J(W^{[1]}, b^{[1]}, \dots, W^{[L]}, b^{[L]}) = \frac{1}{m} \sum_{i=1}^{m} \mathcal{L}(\hat{y}^{(i)}, y^{(i)}) + \frac{\lambda}{2m} \sum_{l=1}^{L} ||W^{[l]}||_F^2$$
  - *Conceptually:* We stack the matrices as one large $(mn, 1)$ vector and calculate $\sqrt{w_1^2 + w_2^2 + \dots}$

**Impact on Backpropagation & Weight Updates**
- **Without Regularization (Old way):**
  $$dW^{[l]} = \text{(from backpropagation)}$$
- **With L2 Regularization (New way):**
  $$dW^{[l]} = \text{(from backpropagation)} + \frac{\lambda}{m} W^{[l]}$$

**Weight Update Step (using learning rate $\alpha$):**
$$
\begin{aligned}
W^{[l]} &= W^{[l]} - \alpha \cdot dW^{[l]} \\
&= W^{[l]} - \alpha \left( \text{(from backpropagation)} + \frac{\lambda}{m} W^{[l]} \right) \\
&= W^{[l]} - \frac{\alpha \lambda}{m} W^{[l]} - \alpha \text{ (from backpropagation)} \\
&= \left( 1 - \frac{\alpha \lambda}{m} \right) W^{[l]} - \alpha \text{ (from backpropagation)}
\end{aligned}
$$

- **Conclusion:** In practice, this penalizes large weights and limits the freedom in your model. The new coefficient $\left( 1 - \frac{\alpha \lambda}{m} \right)$ acts as a penalty that causes the **weight to decay** in proportion to its size during each update.

### Why regularization reduces overfitting?

Here are some intuitions:
  - Intuition 1:
     - If **`lambda` is too large** - a lot of **w's will be close to zeros** which will **make the NN simpler** (you can think of it as it would behave closer to logistic regression).
     - If **`lambda` is good enough** it will just **reduce some weights** that makes the neural network overfit.
  - Intuition 2 (with _tanh_ activation function):
     - If **`lambda` is too large**, **w's will be small** (close to zero) -> **z will become samll** {Z = WX+B} - will use the **linear part of the _tanh_ activation function**, so we will go from non linear activation to _roughly_ linear which would **make the NN a _roughly_ linear classifier**. ans as we know if the activation is liner then nomatter how deep the NN is it's ultimately a Linear regression
     - If `lambda` good enough it will just make some of _tanh_ activations _roughly_ linear which will prevent overfitting.
     
_**Implementation tip**_: if you implement gradient descent, one of the steps to **debug gradient descent** is to **plot the cost function J** as a function of the **number of iterations of gradient descent** and you want to see that the cost function J decreases **monotonically** after every elevation of gradient descent with regularization. If you plot the old definition of J (no regularization) then you might not see it decrease monotonically.

### Dropout Regularization

- In most cases Andrew Ng tells that he uses the L2 regularization.
- The dropout regularization **eliminates some neurons/weights on each iteration based on a probability.**
- Dropout prevents overfitting by ensuring the network does not become overly reliant on any singlevneuron or specific set of weights. It forces the network to learn redundant representations of   data.
- A most common technique to implement dropout is called **"Inverted dropout"**.
- <details>
  <summary>Click to expand!</summary>
  
  ### 1. What is Dropout?
  Dropout is a powerful regularization technique used to reduce variance and prevent overfitting in deep neural networks.
  *   During training, it randomly eliminates (sets to zero) a subset of neurons in a given layer on each iteration.
  *   The probability of a neuron remaining active is defined by the hyperparameter **`keep_prob`** (where $0 \le keep\_prob \le 1$). For example, if `keep_prob = 0.8`, 80% of the neurons stay active, and 20% are dropped.
  
  
  ### 2. Why Dropout Reduces Overfitting
  Even though we randomly shut down parts of the network, the model ultimately performs better on unseen data. This happens for three mathematical reasons:
  
  1.  **Prevents Feature Co-adaptation:** In a standard network, neurons can become highly dependent on the output of one or two specific neurons from the previous layer. With dropout, the network can never rely on any single input feature because that feature might be randomly dropped in the next iteration.
  2.  **Forces Weight Distribution:** Because a neuron cannot depend on a single input, it is forced to spread its weights across all of its inputs. This spreading out of weights naturally shrinks the magnitude of individual weights squared ($||W||^2$), producing an effect mathematically similar to L2 Regularization (Weight Decay).
  3.  **Ensemble Effect:** By dropping different combinations of neurons in every iteration, you are essentially training a large number of smaller, different sub-networks. The final model acts as a combined average (ensemble) of all these smaller networks, leading to a more generalized and robust output.
  
  ---
  
  ### 3. Implementation: "Inverted Dropout"
  The most common and computationally efficient way to implement dropout is called **Inverted Dropout**.
  
  **Implementation Code (for layer 3):**
  ```python
  keep_prob = 0.8   # 0 <= keep_prob <= 1
  l = 3             # targeting layer 3
  
  # 1. Create the boolean mask
  d3 = np.random.rand(a[l].shape[0], a[l].shape[1]) < keep_prob
  
  # 2. Apply the mask to eliminate neurons
  a3 = np.multiply(a3, d3)   
  
  # 3. Scale up the remaining activations (Inverted Dropout Step)
  a3 = a3 / keep_prob        
  ```
  
  #### Step-by-Step Breakdown:
  *   **The Mask ($d3$):** This is a matrix of the exact same shape as the activation matrix $A^{[3]}$. It contains boolean values (1 for True, 0 for False).
  *   **Element-wise Multiplication:** `np.multiply(a3, d3)` applies the mask. If a position in $d3$ is 0, the corresponding neuron in $a3$ becomes 0 (it is "dropped").
  
  ---
  
  ### 4. Deep Dive: Why do we divide by `keep_prob`? (The Scaling Step)
  The critical line of code `a3 = a3 / keep_prob` is what makes this technique "Inverted." This is done to solve a magnitude discrepancy between the Training Phase and the Test Phase.
  
  #### The Problem: Magnitude Discrepancy
  The linear input to the next layer is calculated as $Z^{[l+1]} = W^{[l+1]}A^{[l]} + b^{[l+1]}$.
  1.  **During Training:** If `keep_prob = 0.8`, 20% of the activations in $A^{[l]}$ are zeroed out. The total sum (signal strength) flowing into $Z^{[l+1]}$ decreases by roughly 20%.
  2.  **During Testing:** Dropout is turned **OFF** to maximize prediction accuracy. 100% of the neurons are active. Suddenly, $Z^{[l+1]}$ receives a signal strength that is 20% larger than what it was trained to handle.
  
  This shift in magnitude causes the activation functions in subsequent layers to operate in incorrect ranges, ruining the network's predictions.
  
  #### The Mathematical Solution: Expected Value
  In statistics, the **Expected Value** is the average value a variable takes over time.
  Let $a_i$ be an activation of a single neuron. With dropout, its expected value drops:
  $$E[a_{dropout}] = (keep\_prob \times a_i) + ((1 - keep\_prob) \times 0) = keep\_prob \times a_i$$
  
  To fix this, we artificially boost the 80% of remaining active neurons during training by dividing them by `keep_prob`.
  $$E[a_{scaled}] = keep\_prob \times \left( \frac{a_i}{keep\_prob} \right) = a_i$$
  
  **Conclusion:** By dividing by `keep_prob` during training, the expected value of the activations remains $a_i$. This ensures that when we switch to test time (using 100% of the neurons), the next layer sees the exact same magnitude of data it was trained on. We don't have to write extra scaling code for the testing phase.
  
  ---
  
  ### 5. Exam Checklist: Key Operational Rules
  *   **Iteration-Specific:** A new $d^{[l]}$ mask is generated for **every single pass** (iteration) through the data.
  *   **Consistent Across Propagation:** The exact same mask $d^{[l]}$ used during forward propagation must be cached and used during backpropagation to ensure gradients are only calculated for active neurons.
  *   **Test Time:** Do **not** use dropout at test/inference time. You want deterministic output without added noise. Because of the Inverted Dropout scaling step during training, no weight adjustments are necessary at test time.
</details>
- Code for Inverted dropout:

  ```python
  keep_prob = 0.8   # 0 <= keep_prob <= 1
  l = 3  # this code is only for layer 3
  # the generated number that are less than 0.8 will be dropped. 80% stay, 20% dropped
  d3 = np.random.rand(a[l].shape[0], a[l].shape[1]) < keep_prob

  # This element-wise multiplication ensures that if d3 is 0, the corresponding activation in a3 becomes 0.
  a3 = np.multiply(a3,d3)   

  # increase a3 to not reduce the expected value of output
  # (ensures that the expected value of a3 remains the same) - to solve the scaling problem
  a3 = a3 / keep_prob       
  ```
- Vector d[l] is used for forward and back propagation and is the same for them, but it is different for each iteration (pass) or training example.
- At test time we don't use dropout. If you implement dropout at test time - it would add noise to predictions.

### Understanding Dropout

- In the previous video, the intuition was that dropout randomly knocks out units in your network. So it's as if on every iteration you're working with a smaller NN, and so using a smaller NN seems like it should have a regularizing effect.
- Another intuition: can't rely on any one feature, so have to spread out weights.
- It's possible to show that **dropout has a similar effect to L2 regularization**.
- Dropout can have **different `keep_prob` per layer.**
- The input layer dropout has to be near 1 (or 1 - no dropout) because you don't want to eliminate a lot of features.
- If you're more worried about some layers overfitting than others, you can set a lower `keep_prob` for some layers than others. The downside is, this gives you even more hyperparameters to search for using cross-validation. One other alternative might be to have some layers where you apply dropout and some layers where you don't apply dropout and then just have one hyperparameter, which is a `keep_prob` for the layers for which you do apply dropouts.
- A lot of researchers are using **dropout with Computer Vision (CV)** because **they have a very big input size and almost never have enough data, so overfitting is the usual problem**. And dropout is a regularization technique to prevent overfitting.
- A downside of dropout is that the **cost function J is not well defined** and it will be **hard to debug** (plot J by iteration).
  - To solve that you'll need
    - to turn off dropout, set all the `keep_prob`s to 1,
    - and then run the code and check that it monotonically decreases J and
    - then turn on the dropouts again.

### Other regularization methods

#### Data augmentation:
  - For example in a computer vision data:
    - You can **flip all your pictures horizontally** this will give you **m more data instances.**
    - You could also **apply a random position and rotation to an image** to g**et more data.**
  - For example in OCR, you can impose random rotations and distortions to digits/letters.
  - New data obtained using this technique **isn't as good as the real independent data**, **but still can be used as a regularization technique**.

#### Early stopping:
  - In this technique we **plot the training set** and the **dev set cost together** for **each iteration**. At some iteration the dev set cost will stop decreasing and will start increasing.
  - We will pick the point at which the training set error and dev set error are best (lowest training cost with lowest dev cost).
  - We will take these parameters as the best parameters.
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C2%20-%20Improving%20Deep%20Neural%20Networks%20Hyperparameter%20tuning%2C%20Regularization%20and%20Optimization/Notes/Images/02-_Early_stopping.png?raw=1)
  - Andrew prefers to use L2 regularization instead of early stopping because this technique simultaneously **tries to minimize the cost function** and **not to overfit** which **contradicts the orthogonalization approach** (will be discussed further).
  - But its advantage is that **you don't need to search a hyperparameter** like in other regularization approaches (like `lambda` in L2 regularization).

#### Model Ensembles:
  - Algorithm:
    - Train multiple independent models.
    - At test time average their results.
  - It can get you extra 2% performance.
  - It reduces the generalization error.
  - You can use some snapshots of your NN at the training ensembles them and take the results.


### Normalizing Inputs

- Normalizing your inputs will speed up the training process significantly.

**Normalization Steps**
1. **Get the mean of the training set:**
   $$\mu = \frac{1}{m} \sum_{i=1}^{m} x^{(i)}$$
2. **Subtract the mean from each input:**
   $$X = X - \mu$$
   *(This centers your inputs around 0.)*
3. **Get the variance of the training set:**
   $$\sigma^2 = \frac{1}{m} \sum_{i=1}^{m} (x^{(i)})^2$$
   *(Note: This calculation assumes the mean is now 0 from the previous step.)*
4. **Normalize the variance:**
   $$X = \frac{X}{\sigma^2}$$
   *(Note: In many frameworks, it is also standard to divide by the standard deviation $\sigma$ rather than the variance $\sigma^2$.)*

**Applying Normalization Consistently**
- These steps must be applied to your **training**, **dev (validation)**, and **testing** sets.
- **Crucial Rule:** Always use the $\mu$ and $\sigma^2$ calculated from the *training set* to normalize the dev and test sets. Do not recalculate the mean and variance for the dev or test sets independently.

**Why Normalize?**
- **Unnormalized Inputs:** If features are on entirely different scales, the cost function $J$ will be deep and its shape will be inconsistent (often looking like an elongated, narrow ellipse in a 2D contour plot). Optimizing this takes a long time because gradient descent oscillates heavily and requires a very small learning rate $\alpha$.
- **Normalized Inputs:** The cost function's shape becomes consistent and symmetric (looking like a uniform bowl or a circle in a 2D contour plot). This allows gradient descent to take a much more direct path to the global minimum, enabling you to use a larger learning rate $\alpha$ and making the optimization significantly faster.

### Vanishing / Exploding Gradients

- **Vanishing and Exploding gradients** occur when your derivatives during training become exceptionally small or exceptionally large, making the optimization process extremely difficult.

**Mathematical Intuition**
- To understand the problem, suppose we have a deep neural network with $L$ layers. Assume all activation functions are **linear** ($g(z) = z$) and all biases are zero ($b = 0$).
- The output prediction $\hat{y}$ (or $Y'$) can be written as the product of all weight matrices and the input $X$:
  $$\hat{y} = W^{[L]} W^{[L-1]} \dots W^{[2]} W^{[1]} X$$

- Now, suppose we have 2 hidden units per layer, and our input $X = \begin{bmatrix} 1 \\ 1 \end{bmatrix}$. Let's observe two scenarios for our intermediate weight matrices $W^{[l]}$:

  **1. The Exploding Gradient Case:**
  If the weights are slightly larger than the identity matrix $I$:
  $$W^{[l]} = \begin{bmatrix} 1.5 & 0 \\ 0 & 1.5 \end{bmatrix} \quad (\text{for } l \neq L)$$
  $$\hat{y} = W^{[L]} \begin{bmatrix} 1.5 & 0 \\ 0 & 1.5 \end{bmatrix}^{L-1} X \approx 1.5^L$$
  *Result:* The activation values (and similarly, the gradients) will grow exponentially and **explode** to very large numbers.

  **2. The Vanishing Gradient Case:**
  If the weights are slightly smaller than the identity matrix $I$:
  $$W^{[l]} = \begin{bmatrix} 0.5 & 0 \\ 0 & 0.5 \end{bmatrix} \quad (\text{for } l \neq L)$$
  $$\hat{y} = W^{[L]} \begin{bmatrix} 0.5 & 0 \\ 0 & 0.5 \end{bmatrix}^{L-1} X \approx 0.5^L$$
  *Result:* The activation values (and gradients) will shrink exponentially and **vanish** to near zero.

**Summary of the Problem**
- Activations and gradients scale exponentially as a function of the number of layers $L$.
  - If $W > I$, activations and gradients **explode**.
  - If $W < I$, activations and gradients **vanish**.

**Real-World Context**
- Modern neural networks can be incredibly deep (e.g., Microsoft's ResNet with 152 layers).
- If gradients decrease exponentially as a function of $L$, gradient descent will take microscopic steps in the earlier layers. It will take an impossibly long time for the network to learn anything useful.
- Conversely, exploding gradients can cause numerical overflows, resulting in `NaN` (Not a Number) values during training.

**The Solution**
- While it doesn't solve the problem with 100% perfection, a highly effective partial solution is the **careful initialization of weights** (e.g., Xavier/He initialization).

### Weight Initialization for Deep Networks

- A partial solution to the **Vanishing / Exploding Gradients** problem in Neural Networks is a more careful choice of the random initialization of weights.
- In a single neuron (Perceptron model), the pre-activation is:
  $$Z = w_1x_1 + w_2x_2 + \dots + w_nx_n$$
  - If $n$ (number of input features) is very large, we want the weights $W$ to be smaller to prevent the sum $Z$ from exploding.
- It turns out that we need the variance of $W$ to equal $\frac{1}{n}$ to maintain a stable range for $Z$.

**Initialization Formulas**
- **Xavier / Glorot Initialization** (Best used with `tanh` activation):
  ```python
  W = np.random.randn(shape) * np.sqrt(1 / n[l-1])

### Gradient Checking (Numerical Differentiation)



#### Details

### 1. What is Gradient Checking?
Gradient checking is a "sanity check" used to verify that your manually coded **Backpropagation** (the analytical gradient) is mathematically correct. It uses a numerical approximation of the derivative to ensure your code isn't suffering from subtle mathematical bugs.

### 2. Why & When is it needed?
*   **The Problem:** Backpropagation is complex. A tiny typo (like a misplaced minus sign or a missing term) can lead to a model that "runs" without errors but never actually converges to the optimal solution.
*   **The Solution:** You compare your code’s output ($d\theta_{approx}$) with a numerical "ground truth" ($d\theta_{grad}$).
*   **When to use:**
    *   Only during **debugging/development**.
    *   Never during actual training (it is extremely slow because it requires computing the cost function twice for every single parameter).

---

### 3. What are we basically doing?
We are using the geometric definition of a derivative. If we want to find the slope of the cost function $J$ at a specific point $\theta$, we move a tiny amount $\epsilon$ (epsilon) away from $\theta$ and see how much the cost changes.

---

### 4. Method Comparison: One-Sided vs. Two-Sided
To find the gradient numerically, we have two options:

#### Option A: One-Sided Difference
This looks at the change from the current point to a point slightly ahead.
$$\text{Formula: } \frac{J(\theta + \epsilon) - J(\theta)}{\epsilon}$$
*   **Accuracy:** Moderate. It follows the standard limit definition of a derivative.

#### Option B: Two-Sided Difference (The "Andrew Ng" Way)
This "brackets" the point $\theta$ by looking slightly behind and slightly ahead.
$$\text{Formula: } \frac{J(\theta + \epsilon) - J(\theta - \epsilon)}{2\epsilon}$$
*   **Accuracy:** **Significantly Higher.** This is the preferred method for Gradient Checking.

---

### 5. Why is Two-Sided "Better"? (The Order of Error)
In numerical analysis, "better" means the **error is smaller**. We use Taylor Series Expansion to prove why the Two-Sided version is superior.

#### The Error of One-Sided: $O(\epsilon)$
When we expand the math for the one-sided formula, the leftover error term is proportional to $\epsilon$.
*   If $\epsilon = 10^{-7}$, your error is roughly **$0.0000001$**.

#### The Error of Two-Sided: $O(\epsilon^2)$
In the two-sided formula, the Taylor expansion shows that the first-order error terms actually **cancel each other out**, leaving an error term proportional to $\epsilon^2$.
*   If $\epsilon = 10^{-7}$, your error is $(10^{-7})^2 = 10^{-14}$.
*   Your error becomes **$0.00000000000001$**.

> **Summary:** The Two-Sided formula is $1,000,000$ times more accurate than the One-Sided formula when using a small epsilon. This precision is necessary because when checking gradients, we need to distinguish between a "tiny numerical rounding error" and a "math bug in our code."

---
### 6. Implementation
1.  **Reshape Parameters:** Take all your weight matrices $W^{[L]}$ and bias vectors $b^{[L]}$ and concatenate (reshape) them into one large vector called $\theta$.
2.  **Reshape Gradients:** Similarly, take all your computed gradients $dW^{[L]}$ and $db^{[L]}$ and concatenate them into one large vector called $d\theta$.
3.  **Define the Cost Function:** You now have a function $J(\theta)$ that takes this single large vector and returns the cost.

#### The Approximation Algorithm
For each element $i$ in the vector $\theta$:
```python
epsilon = 1e-7   # A very small value
for i in range(len(theta)):
    # Create two temporary copies to shift a single parameter
    theta_plus = np.copy(theta)
    theta_plus[i] = theta_plus[i] + epsilon
    
    theta_minus = np.copy(theta)
    theta_minus[i] = theta_minus[i] - epsilon
    
    # Calculate the numerical gradient for this specific parameter
    d_theta_approx[i] = (J(theta_plus) - J(theta_minus)) / (2 * epsilon)
```
---
### 7. How to interpret the results?
After calculating the numerical gradient ($d\theta_{approx}$) and your backprop gradient ($d\theta$), you compute the **relative difference**:

$$\text{Difference} = \frac{\|d\theta_{approx} - d\theta\|_2}{\|d\theta_{approx}\|_2 + \|d\theta\|_2}$$

| Difference Value | Conclusion |
| :--- | :--- |
| **$10^{-7}$ or smaller** | **Great!** Your backprop is likely perfect. |
| **$10^{-5}$** | **Careful.** There might be a subtle bug; double-check the math. |
| **$10^{-3}$ or larger** | **Bug Detected.** Something is wrong in your backpropagation code. |

---

**Practical Tip for your Research:** Most modern frameworks like PyTorch or JAX have a built-in `gradcheck` function that uses this exact Two-Sided logic. When you start writing custom layers for your Vision-Language Model (VLM) thesis, running a `gradcheck` on your new layer is the first thing you should do!

Since you're looking into Computer Vision research, are you planning to implement any custom loss functions for your thesis, or are you sticking to standard ones for now?

### Initialization summary

- The weights W<sup>[l]</sup> should be initialized randomly to break symmetry

- It is however okay to initialize the biases b<sup>[l]</sup> to zeros. Symmetry is still broken so long as W<sup>[l]</sup> is initialized randomly

- Different initializations lead to different results

- Random initialization is used to break symmetry and make sure different hidden units can learn different things

- Don't intialize to values that are too large

- He initialization works well for networks with ReLU activations.

### Regularization summary

#### 1. L2 Regularization   
**Observations**:   
  - The value of λ is a hyperparameter that you can tune using a dev set.
  - L2 regularization makes your decision boundary smoother. If λ is too large, it is also possible to "oversmooth", resulting in a model with high bias.

**What is L2-regularization actually doing?**:   
  - L2-regularization relies on the assumption that a model with small weights is simpler than a model with large weights. Thus, by penalizing the square values of the weights in the cost function you drive all the weights to smaller values. It becomes too costly for the cost to have large weights! This leads to a smoother model in which the output changes more slowly as the input changes.

**What you should remember:**   
Implications of L2-regularization on:
  - cost computation:
    - A regularization term is added to the cost
  - backpropagation function:
    - There are extra terms in the gradients with respect to weight matrices
  - weights:
    - weights end up smaller ("weight decay") - are pushed to smaller values.
    
#### 2. Dropout   
**What you should remember about dropout:**   
- Dropout is a regularization technique.
- You only use dropout during training. Don't use dropout (randomly eliminate nodes) during test time.
- Apply dropout both during forward and backward propagation.
- During training time, divide each dropout layer by keep_prob to keep the same expected value for the activations. For example, if `keep_prob` is 0.5, then we will on average shut down half the nodes, so the output will be scaled by 0.5 since only the remaining half are contributing to the solution. Dividing by 0.5 is equivalent to multiplying by 2. Hence, the output now has the same expected value. You can check that this works even when keep_prob is other values than 0.5.


## Optimization algorithms

### **Mini-batch gradient descent**



- Training NN with a large data is slow. So to find an optimization algorithm that runs faster is a good idea.
- Suppose we have `m = 50 million`. To train this data it will take a huge processing time for one step.
  - because 50 million won't fit in the memory at once we need other processing to make such a thing.
- It turns out you can make a faster algorithm to make gradient descent process some of your items even before you finish the 50 million items.
- Suppose we have split m to **mini batches** of size 1000.
  - `X{1} = 0    ...  1000`
  - `X{2} = 1001 ...  2000`
  - `...`
  - `X{bs} = ...`
- We similarly split `X` & `Y`.
- So the definition of mini batches ==> `t: X{t}, Y{t}`
- In **Batch gradient descent** we run the gradient descent on the whole dataset.
- While in **Mini-Batch gradient descent** we run the gradient descent on the mini datasets.
- Mini-Batch algorithm pseudo code:
  ```
  for t = 1:No_of_batches                         # this is called an epoch
  	AL, caches = forward_prop(X{t}, Y{t})
  	cost = compute_cost(AL, Y{t})
  	grads = backward_prop(AL, caches)
  	update_parameters(grads)
  ```
- The code inside an epoch should be vectorized.
- Mini-batch gradient descent works much faster in the large datasets.

#### Understanding mini-batch gradient descent

- In **Batch Gradient Descent**, the cost $J$ should decrease on every single iteration. If it doesn't, your learning rate $\alpha$ is likely too high.
- In **Mini-Batch Gradient Descent**, the cost function is noisier. Because each step is calculated on a different subset of data, the cost may "wiggle"—it might go up slightly on some iterations but should trend downward over time.
<br>

#### **Comparing Batch Sizes**

The choice of mini-batch size determines the behavior of the optimization:

| Type | Mini-Batch Size | Description |
| :--- | :--- | :--- |
| **Batch Gradient Descent** | size $= m$ | Processes the entire dataset at once. |
| **Stochastic Gradient Descent (SGD)** | size $= 1$ | Processes only one example per step. |
| **Mini-Batch Gradient Descent** | $1 <$ size $< m$ | A balance between the two extremes. |

- Mini-batch gradient descent:
  1. faster learning:
      - you have the vectorization advantage
      - make progress without waiting to process the entire training set
  2. doesn't always exactly converge (oscelates in a very small region, but you can reduce learning rate)
- Guidelines for choosing mini-batch size:
  1. If small training set (< 2000 examples) - use batch gradient descent.
  2. It has to be a power of 2 (because of the way computer memory is layed out and accessed, sometimes your code runs faster if your mini-batch size is a power of 2):
    `64, 128, 256, 512, 1024, ...`
  3. Make sure that mini-batch fits in CPU/GPU memory.
- Mini-batch size is a `hyperparameter`.



### **Exponentially Weighted Averages (EWA)**



Exponentially Weighted Averages (also known as **Exponentially Weighted Moving Averages** in statistics) are a fundamental building block for advanced optimization algorithms like Momentum, RMSProp, and Adam. They provide a way to smooth out noisy data and identify trends by giving more weight to recent observations while exponentially "forgetting" older ones.



#### 1. The Core Formula
The average $V$ at time $t$ is calculated using the current observation $\theta_t$ and the previous average $V_{t-1}$:

$$V_t = \beta V_{t-1} + (1 - \beta) \theta_t$$

*   **$V_t$**: The current moving average (an estimate of the trend).
*   **$\theta_t$**: The actual data point at time $t$ (e.g., today's temperature).
*   **$\beta$**: The **forgetting factor** (hyperparameter), typically between 0 and 1.
*   **$V_0$**: Usually initialized to **0**.

<br>

#### 2. Understanding the Hyperparameter $\beta$
The value of $\beta$ determines how many previous data points significantly influence the current average. A useful rule of thumb is that $V_t$ averages over approximately **$\frac{1}{1 - \beta}$** days/entries.

| $\beta$ Value | Window Size | Effect |
| :--- | :--- | :--- |
| **0.9** | ~10 days | A balanced average; follows trends with moderate noise. |
| **0.98** | ~50 days | **Very smooth**; less noisy, but has **high latency** (shifts the curve to the right and reacts slowly to changes). |
| **0.5** | ~2 days | **Very noisy**; reacts almost instantly to changes but is susceptible to outliers. |
<br>



#### 3. Mathematical Intuition: Why "Exponential"?
If we expand the formula for $V_{100}$, we see that it is aweighted sum of all previous days:
$$V_{100} = 0.1\theta_{100} + 0.1(0.9)\theta_{99} + 0.1(0.9)^2\theta_{98} + 0.1(0.9)^3\theta_{97} + \dots$$

*   **Exponential Decay:** The weights decrease exponentially as we go back in time.
*   **The $1/e$ Rule:** The weight drops to about **35%** (specifically $1/e$) of its initial value after $\frac{1}{1-\beta}$ steps. This is why we consider that window the "effective" range of the average.
<br>



#### 4. Why Use EWA in Deep Learning?
In optimization, Gradient Descent often "oscillates" (zig-zags) toward the minimum. EWA helps by:
1.  **Averaging out oscillations:*\\* The vertical fluctuations cancel out, while the horizontal progress is maintained.
2.  **Computational Efficiency:** Unlike a "Simple Moving Average" (which requires storing the last $N$ data points in memory), EWA only requires **one line of code** and **one variable** in memory.

**Implementation (Python-style logic):**
```python
v = 0
for theta in data:
    v = beta * v + (1 - beta) * theta
```
<br>



#### 5. Summary of Trade-offs
*   **High $\beta$:** Provides a smoother curve but is slower to adapt to actual changes in data (higher latency).
*   **Low $\beta$:** Adapts quickly to changes but results in a "wiggly" curve that captures too much noise.
*   **Accuracy vs. Efficiency:** While a true moving window average (summing $N$ days and dividing by $N$) is more mathematically "accurate," EWA is preferred in Deep Learning because it is significantly faster and uses almost no memory.

### **Bias Correction in Exponentially Weighted Averages**



While the standard EWA formula is efficient, it suffers from a significant flaw during the initial steps of a sequence: it starts off extremely low. **Bias Correction** is the technical adjustment used to fix this "cold start" problem and make the early estimates more accurate.

<br>


#### The Problem: The "Cold Start" Bias
Because we initialize $V_0 = 0$, the first few values of $V_t$ are heavily weighted toward zero rather than the actual data.

**Example:**
If we set $\beta = 0.98$ and the first data point $\theta_1 = 40$:
*   **Standard Formula:** $V_1 = 0.98(0) + 0.02(40) = \mathbf{0.8}$
*   **The Reality:** The temperature is $40$, but our average says it's $0.8$.

This creates a "purple curve" effect (from the lecture) where the estimate takes a long time to "warm up" and reach the actual trend of the data.

<br>

#### The Solution: The Correction Formula
To compensate for this initial dip, we divide the average $V_t$ by a correction factor that accounts for how many steps have passed:

$$V_t^{corrected} = \frac{V_t}{1 - \beta^t}$$

*   **$t$**: The current time step (day 1, day 2, etc.).
*   **$\beta^t$**: $\beta$ raised to the power of the current time step.

<br>

#### Why This Works
The correction factor $1 - \beta^t$ effectively "scales up" the estimate during the early stages and naturally phases itself out over time.

*   **At the Start ($t$ is small):**
    When $t=1$, the denominator is $1 - \beta$. In our example where $\beta=0.98$, the denominator is $0.02$.
    Dividing our biased $V_1$ ($0.8$) by $0.02$ gives us **$40$**—the exact, unbiased temperature of the first day.
*   **As Time Passes ($t$ is large):**
    As $t$ increases, $\beta^t$ becomes a very small number approaching **0**. Consequently, $1 - \beta^t$ approaches **1**.
    At this stage, the correction factor has no effect, and the formula reverts to the standard EWA.
<br>


#### Implementation in Machine Learning
*   **Industry Practice:** In many deep learning implementations, researchers often **omit** bias correction. This is because most neural networks are trained for thousands of iterations; the initial bias only affects the first few dozen steps, which becomes negligible in the long run.
*   **When to use it:** Use it if you need highly accurate averages from the very first iteration or if you are using a very high $\beta$ (like $0.999$), where the warm-up period lasts much longer.

<br>


#### Summary Table
| Feature | Without Bias Correction | With Bias Correction |
| :--- | :--- | :--- |
| **Start Value** | Starts at 0; very inaccurate early on. | Corrected to match initial data points. |
| **Late Value** | Accurate. | Accurate (Correction factor becomes ~1). |
| **Complexity** | One line of code. | Slightly more math (needs $t$). |
| **Visual** | Curve starts far below the data. | Curve follows data from $t=1$. |